# Kaggriculture — Balanced Farmer v1

## Baseline Autonomous Farming Agent

### Project Report

---

## 1. Executive Summary

**Balanced Farmer v1** is a rule-based autonomous agent developed for the **Kaggriculture** competition.

The competition is a two-player farming simulation in which each agent manages a farm over **30 in-game days / 720 turns**. The objective is simple:

> **Finish the season with more money in the bank than the opponent.**

Our first model is intentionally designed as a **baseline model**, rather than as a fully optimized strategy.

The model focuses on five core principles:

1. **Generate early cash flow using wheat.**
2. **Generate high-value revenue using melons.**
3. **Keep crops alive through systematic watering.**
4. **Harvest and sell production while avoiding unnecessary market crashes.**
5. **Reinvest profits into seeds, labor, and eventually land expansion.**

The model deliberately excludes sophisticated components such as livestock optimization, fertilizer optimization, opponent modeling, and mathematical market optimization. These can be introduced later and evaluated against this baseline.

---

# 2. Competition Objective

Kaggriculture is a turn-based economic simulation.

Each player controls a farmer and can potentially hire temporary farm hands. The farm can contain crops, animal structures, weeds, and empty land.

The season consists of:

| Parameter             |            Value |
| --------------------- | ---------------: |
| Players               |                2 |
| Season length         |          30 days |
| Turns per day         |               24 |
| Total turns           |              720 |
| Starting money        |           $3,000 |
| Initial land          | 1 × 5×5 quadrant |
| Full farm             |            10×10 |
| Initial unlocked land |              25% |
| Shed capacity         |        100 items |

Every turn, the agent can issue:

* Farmer actions
* Farm-hand actions
* Market actions

The final winner is the player with the **highest bank balance** at the end of the season.

Importantly:

> **Unsold inventory does not count toward the final score.**

Therefore, production alone is not sufficient. Production must eventually be converted into cash.

---

# 3. Model Philosophy

Our model is based on a deliberately simple economic philosophy:

> **Build a stable cash-flow engine first, then add higher-return opportunities.**

Instead of immediately attempting to optimize every aspect of the environment, we separate the problem into manageable components.

The initial model is therefore a **rule-based control system**.

It does not use:

* Reinforcement learning
* Neural networks
* Monte Carlo tree search
* Genetic algorithms
* Linear programming
* Dynamic programming
* Opponent prediction
* Learned market forecasting

This is intentional.

The baseline should be:

* deterministic
* interpretable
* reproducible
* computationally cheap
* easy to debug
* easy to improve

---

# 4. Core Economic Strategy

The model uses two crops:

### Wheat

Wheat is our **stable production asset**.

Its advantages are:

* Seed cost: **$10**
* Short production cycle
* Cheap to restart
* Provides relatively reliable cash flow
* Can later be used as animal feed

Wheat therefore provides the model with liquidity.

### Melon

Melon is our **high-value production asset**.

Its advantages include:

* Seed cost: **$80**
* Base market price: **$250**
* Maximum yield: **6**
* High potential revenue per occupied tile

However, melon has a major weakness:

> Its market price is highly sensitive to oversupply.

Therefore, melon is treated as the model's higher-risk, higher-value crop.

---

# 5. Portfolio Structure

The model effectively creates a very simple agricultural portfolio:

```text
             BALANCED FARM
                   │
          ┌────────┴────────┐
          │                 │
        WHEAT              MELON
          │                 │
       Stability          Upside
          │                 │
    Cash generation    High-value sales
          │                 │
          └────────┬────────┘
                   │
                Revenue
                   │
              Reinvestment
```

This is preferable to the tutorial's pure-melon strategy because we don't want the entire farm to depend on one volatile product.

---

# 6. Production Cycle

The model follows a repeating production cycle:

```text
Buy seeds
   ↓
Plant
   ↓
Water
   ↓
Wait for maturity
   ↓
Harvest
   ↓
Store
   ↓
Sell
   ↓
Cash
   ↓
Buy more seeds
   ↓
Repeat
```

The objective is to keep the farm producing continuously throughout the season.

---

# 7. Crop Maintenance

Crop survival is given the highest operational priority.

The model checks whether a plant has been watered that day.

If:

```python
tile["watered_today"] == False
```

the farmer attempts to:

```text
WATER
```

before doing lower-priority activities.

This is important because crops can become weeds after consecutive days without watering.

Therefore:

> **Preventing production loss comes before expanding production.**

---

# 8. Farmer Priority System

The farmer follows a simple hierarchy.

### Priority 1 — Water

If a nearby plant needs watering:

```text
WATER
```

### Priority 2 — Harvest wheat

If mature wheat has harvestable yield:

```text
HARVEST
```

### Priority 3 — Harvest melon

If mature melon has harvestable yield:

```text
HARVEST
```

### Priority 4 — Plant melon

If a melon seed is available and the farmer is standing on an empty tile:

```text
PLANT MELON
```

### Priority 5 — Plant wheat

If no melon seed is available but wheat seed exists:

```text
PLANT WHEAT
```

### Priority 6 — Move

If there is no immediate action, the farmer moves toward the nearest useful tile.

### Priority 7 — PASS

If there is nothing useful to do:

```text
PASS
```

---

# 9. Target Selection

The agent scans the farm and identifies useful tiles.

Potential targets include:

* plants requiring water
* mature wheat
* mature melon
* empty tiles for planting

The model calculates Manhattan distance:

$$
d = |x_f-x_t| + |y_f-y_t|
$$

where:

* \(x_f,y_f\) = farmer position
* \(x_t,y_t\) = target position

The nearest useful target is selected.

This gives the baseline a basic form of spatial planning without requiring sophisticated pathfinding.

---

# 10. Movement

The movement system is deliberately simple.

If the farmer is west of the target:

```text
EAST
```

If the farmer is east of the target:

```text
WEST
```

If the farmer is north of the target:

```text
SOUTH
```

If the farmer is south of the target:

```text
NORTH
```

The model therefore performs greedy movement toward the nearest useful task.

It does **not** currently perform global route optimization.

---

# 11. Market Strategy

The market is one of the most important components of Kaggriculture.

The model recognizes two important facts:

1. Production has no value if it never becomes cash.
2. Selling too much at once can reduce the price.

Therefore, the model treats wheat and melon differently.

---

# 12. Wheat Selling Policy

Wheat is treated as a relatively stable cash-flow product.

The model sells wheat when:

```text
Wheat price >= $25
```

The baseline therefore does not attempt sophisticated wheat speculation.

Conceptually:

```text
Wheat harvested
      ↓
Wheat in shed
      ↓
Price >= $25?
   /       \
 YES       NO
  ↓         ↓
SELL       HOLD
```

Because the base wheat price is approximately $25, this essentially means the model is willing to convert wheat into cash around its normal value.

---

# 13. Melon Selling Policy

Melon receives more conservative treatment.

The model uses:

```text
MELON_SELL_THRESHOLD = $200
```

Therefore:

```text
if melon price >= $200:
    sell
else:
    hold
```

The reasoning is that melon has a high nominal value but can experience severe price deterioration when the market becomes oversupplied.

---

# 14. Batch Selling

The model does not sell all available melons simultaneously.

Instead:

```text
MELON_SELL_BATCH = 5
```

At most five melons are submitted in a single sell decision.

Conceptually:

```text
20 melons in shed
       ↓
Price >= $200
       ↓
Sell 5
       ↓
Re-evaluate next turn
       ↓
Sell another 5 if conditions remain favorable
```

This is a primitive form of **market-impact management**.

It is intentionally simple.

Later versions can improve this substantially by explicitly modeling the market's price function.

---

# 15. Seed Purchasing

The agent does not purchase enormous quantities of seeds.

Instead, it uses a simple inventory rule:

> **If seed inventory reaches zero and sufficient cash remains, purchase another seed.**

For wheat:

```text
If wheat seeds = 0
and cash >= seed cost + cash reserve:
    BUY_SEED WHEAT 1
```

For melon:

```text
If melon seeds = 0
and cash >= seed cost + cash reserve:
    BUY_SEED MELON 1
```

This creates a continuous production loop without tying up excessive capital in unused seeds.

---

# 16. Cash Reserve

The model maintains a minimum cash reserve of:

```text
$500
```

The intention is to prevent the farm from becoming completely illiquid.

Instead of:

```text
Cash = $3,000
 ↓
Spend everything
 ↓
Cash = $0
```

we use:

```text
Cash
 ↓
Production investment
 ↓
Keep approximately $500 available
```

This is a basic form of **working-capital management**.

---

# 17. Farm Expansion

The initial farm only covers one 5×5 quadrant.

The model therefore has a simple expansion policy.

### First expansion

```text
NE
```

when cash reaches:

```text
$4,000
```

### Second expansion

```text
SW
```

when cash reaches:

```text
$8,000
```

### Third expansion

```text
SE
```

when cash reaches:

```text
$12,000
```

Thus:

```text
Starting
   │
   ▼
NW
   │
Cash >= $4k
   ▼
NW + NE
   │
Cash >= $8k
   ▼
NW + NE + SW
   │
Cash >= $12k
   ▼
Full 10×10 farm
```

The thresholds are **baseline heuristics**, not claims of optimality.

They give us a clean experimental starting point.

---

# 18. Farm Hands

The model introduces limited labor expansion.

Farm hands cost more with successive hires during the same day, following the Fibonacci-based cost structure.

To avoid unnecessary complexity, the baseline limits itself to **one farm hand**.

The hand becomes eligible for hiring when:

```text
Active plants >= 15
```

and no hand has already been hired that day.

The logic is:

> Don't pay for labor until there is enough agricultural workload to justify it.

---

# 19. Farm-Hand Role

The farm hand does not have an independent economic strategy.

Instead, it acts as a maintenance assistant.

Its priorities are:

1. Water plants
2. Harvest mature wheat
3. Harvest mature melon
4. Move toward the next useful tile
5. PASS

This keeps the first version simple.

There is no sophisticated division of labor such as:

```text
Farmer → planting
Hand 1 → watering
Hand 2 → harvesting
Hand 3 → logistics
```

That is a potential future improvement.

---

# 20. What the Model Does NOT Do

A major feature of this baseline is what it deliberately leaves out.

### No livestock

The model does not currently purchase:

* geese
* cows
* sheep

### No fertilizer

The model does not:

* collect fertilizer
* buy fertilizer
* optimize fertilizer application

### No advanced market forecasting

The model does not estimate future prices.

### No opponent modeling

It does not inspect the opponent's farm to infer their strategy.

### No dynamic crop allocation

It does not dynamically determine the economically optimal crop for every tile.

### No sophisticated labor scheduling

Farm hands simply assist with nearby farming tasks.

### No mathematical optimization

There is no LP, DP, MCTS, RL, or learned policy.

These omissions are deliberate.

---

# 21. Why Start Without Animals?

Animals introduce another production system.

For example:

```text
Buy animal
     ↓
Build structure
     ↓
Place animal
     ↓
Feed daily
     ↓
Care daily
     ↓
Wait for production
     ↓
Harvest
     ↓
Collect fertilizer
     ↓
Sell
```

They also require wheat as feed.

Including animals immediately would make it difficult to determine **which component actually improved the strategy**.

Therefore:

> Animals are reserved for a later experimental version.

---

# 22. Why Start Without Fertilizer?

Fertilizer can increase crop yields, but it introduces another resource-allocation problem.

We first want to measure:

> **How profitable is the basic farming system without fertilizer?**

Then we can add fertilizer and measure the incremental improvement.

For example:

```text
Balanced Farmer v1
        ↓
+ Fertilizer
        ↓
Measure Δ profit
```

This produces a much cleaner experimental methodology.

---

# 23. Agent Architecture

The implementation is divided into several logical components.

```text
                    balanced_farmer()
                           │
          ┌────────────────┼────────────────┐
          │                │                │
          ▼                ▼                ▼
      Market          Farm Logic       Labor Logic
          │                │                │
          ▼                ▼                ▼
    Buy / Sell        Water / Plant     Hire
    Seeds             Harvest / Move
          │                │
          └────────┬───────┘
                   ▼
                Action
                   │
                   ▼
             Kaggriculture
```

### Main components

| Component                     | Function             |
| ----------------------------- | -------------------- |
| `_step_toward()`              | Movement             |
| `_manhattan_distance()`       | Distance calculation |
| `_count_plants()`             | Workload estimation  |
| `_find_nearest_target()`      | Task selection       |
| `_find_best_farming_target()` | Farmer task priority |
| `_market_actions()`           | Buying/selling       |
| `_land_actions()`             | Expansion            |
| `_should_hire()`              | Labor decision       |
| `_hand_action()`              | Farm-hand control    |
| `balanced_farmer()`           | Main policy          |
| `agent()`                     | Kaggle entry point   |

---

# 24. Decision-Making Loop

Every turn effectively follows this process:

```text
             OBSERVATION
                  │
                  ▼
        Read current farm state
                  │
       ┌──────────┴──────────┐
       │                     │
       ▼                     ▼
  Market state           Farm state
       │                     │
       ▼                     ▼
 Buy / Sell            Water / Harvest
 Seeds                 / Plant / Move
       │                     │
       └──────────┬──────────┘
                  ▼
          Land / Labor check
                  │
                  ▼
             Final actions
                  │
                  ▼
               GAME
```

---

# 25. Operational Priority

The model's overall priority hierarchy is:

```text
1. Keep existing production alive
2. Harvest available production
3. Plant available land
4. Convert production into cash
5. Maintain seed supply
6. Add labor when workload justifies it
7. Expand land when financially justified
8. PASS when no productive action exists
```

This represents a basic **operations-management philosophy**:

> Protect existing productive assets before committing capital to additional capacity.

---

# 26. Economic Interpretation

Balanced Farmer v1 can be viewed as a simplified production system.

### Inputs

* Starting capital
* Seeds
* Land
* Farmer labor
* Farm-hand labor

### Production process

```text
Land + Seeds + Labor + Water
             ↓
          Crops
```

### Outputs

```text
Wheat
Melon
```

### Revenue

```text
Crops
  ↓
Market
  ↓
Cash
```

### Reinvestment

```text
Cash
 ├── Seeds
 ├── Labor
 └── Land
```

This makes the competition fundamentally a problem of **capital allocation under dynamic prices and resource constraints**.

---

# 27. Main Risks

The baseline faces several important risks.

### Risk 1 — Market crash

Excessive melon selling can cause the price to fall.

### Risk 2 — Farmer inefficiency

One farmer may not be able to maintain a large farm.

### Risk 3 — Land underutilization

Buying land without enough labor or seeds can waste capital.

### Risk 4 — Crop neglect

Failure to water can destroy production.

### Risk 5 — Liquidity shortage

Overspending on seeds, labor, or land can leave insufficient operating capital.

### Risk 6 — Shed overflow

The shed has a capacity of only 100 non-seed items.

Excess inventory can be discarded.

---

# 28. Important Baseline Assumptions

The strategy assumes:

1. Wheat provides useful early cash flow.
2. Melon provides attractive high-value production.
3. Holding melons during weak prices is better than immediately selling them.
4. One farm hand becomes useful once the crop count becomes sufficiently large.
5. Land should be purchased only after meaningful capital accumulation.
6. Maintaining a cash reserve improves robustness.
7. Simple greedy movement is adequate for a first implementation.

None of these should be treated as proven optimal.

They are **hypotheses that the baseline allows us to test**.

---

# 29. Baseline Strengths

Balanced Farmer v1 should have several advantages over the tutorial's simple Melon Maxxer.

### Diversification

It does not depend entirely on melon.

### Cash management

It maintains a cash reserve.

### Market awareness

It does not blindly dump all melons.

### Labor scaling

It can introduce a farm hand.

### Land scaling

It can expand beyond the initial quadrant.

### Maintainability

The model is composed of small, understandable rules.

### Experimental usefulness

Each component can be independently improved and tested.

---

# 30. Baseline Weaknesses

The strategy is still intentionally naive.

### Weakness 1 — Static thresholds

The values:

```text
$200 melon sell threshold
$4k / $8k / $12k land thresholds
15 plants for hiring
$500 cash reserve
```

are manually chosen.

They are not optimized.

### Weakness 2 — No price-curve calculation

The game provides a detailed market-price function, but v1 does not exploit it.

### Weakness 3 — No marginal-profit calculation

The model doesn't calculate:

$$
\text{Marginal Revenue} - \text{Marginal Cost}
$$

before making investments.

### Weakness 4 — No crop optimization

The crop mix is fixed rather than dynamically optimized.

### Weakness 5 — No livestock

Potentially profitable animal production is ignored.

### Weakness 6 — No opponent interaction

The opponent's visible farm is not used strategically.

### Weakness 7 — Greedy movement

The farmer can waste turns moving between distant targets.

---

# 31. Experimental Roadmap

The most important purpose of this model is to create a **controlled starting point**.

We can develop the agent incrementally.

### Version 1 — Balanced Farmer

```text
Wheat
+
Melon
+
Basic market rules
+
Basic labor
+
Basic expansion
```

### Version 2 — Dynamic Market

Replace fixed thresholds with the actual market-price function.

```text
Inventory
+
Price
+
Expected future demand
        ↓
Optimal selling quantity
```

### Version 3 — Fertilizer

Introduce:

```text
Animal fertilizer
+
Purchased fertilizer
+
Crop yield optimization
```

### Version 4 — Livestock

Add:

```text
Goose
Cow
Sheep
```

and compare their production economics.

### Version 5 — Dynamic Crop Allocation

Instead of:

```text
Wheat + Melon
```

the agent evaluates:

```text
Wheat
Carrot
Tomato
Strawberry
Melon
```

and determines which crop has the highest expected economic return.

### Version 6 — Labor Optimization

Optimize:

```text
When to hire
How many to hire
Which task each hand performs
```

### Version 7 — Land Optimization

Determine when purchasing:

```text
NE
SW
SE
```

actually produces a positive return on invested capital.

### Version 8 — Opponent-Aware Market Strategy

Use the publicly visible opponent farm to estimate:

```text
Opponent production
        ↓
Expected market supply
        ↓
Expected price pressure
        ↓
Optimal crop strategy
```

### Version 9 — Full Economic Agent

Ultimately:

```text
                  FULL AGENT
                      │
     ┌────────────────┼────────────────┐
     │                │                │
  Production        Labor           Market
     │                │                │
     ├── Crops        ├── Farmer      ├── Prices
     ├── Animals      ├── Hands       ├── Demand
     ├── Fertilizer   └── Scheduling  └── Selling
     │
     └───────────────┬────────────────┘
                     │
                  Capital
                     │
              Land Expansion
                     │
                     ▼
                 PROFIT
```

---

# 32. Experimental Methodology

We should not judge improvements from a single game.

Because the environment contains randomness and the competition is head-to-head, we should evaluate each version across many episodes.

For each experiment, record:

| Metric            | Purpose                  |
| ----------------- | ------------------------ |
| Final cash        | Primary economic outcome |
| Win rate          | Competitive performance  |
| Average profit    | Expected performance     |
| Median profit     | Robustness               |
| Worst-case profit | Downside risk            |
| Wheat produced    | Production efficiency    |
| Melon produced    | Production efficiency    |
| Selling price     | Market execution         |
| Land purchased    | Capital allocation       |
| Hands hired       | Labor utilization        |
| Crop survival     | Operational reliability  |
| Unused inventory  | Resource waste           |

The key metric remains:

> **Final bank balance / win rate against comparable opponents.**

---

# 33. Ablation Testing

A particularly useful method will be **ablation testing**.

For example:

### Experiment A

```text
Wheat + Melon
```

### Experiment B

```text
Wheat + Melon
+ batch selling
```

Compare A vs B.

Then:

```text
B + land expansion
```

Then:

```text
C + farm hand
```

Then:

```text
D + fertilizer
```

This allows us to determine the **marginal contribution** of each strategic component.

---

# 34. Baseline Success Criterion

The purpose of Balanced Farmer v1 is not necessarily to reach the leaderboard immediately.

Its primary success criterion is:

> **Provide a stable benchmark that future strategies can beat.**

For example, suppose we obtain:

```text
Balanced Farmer v1
Win rate = 55%
```

Then a new market optimizer produces:

```text
Win rate = 63%
```

We have evidence that the market optimizer adds value.

Without a stable baseline, we cannot make that comparison confidently.

---

# 35. Final Model Specification

### Model name

**Balanced Farmer v1**

### Model type

**Deterministic rule-based autonomous agent**

### Production

```text
Wheat + Melon
```

### Watering

```text
Daily maintenance
```

### Harvesting

```text
Harvest mature wheat/melon
```

### Market

```text
Wheat:
    sell around base price

Melon:
    sell only >= $200
    sell in batches of 5
```

### Seeds

```text
Purchase one seed when inventory reaches zero
```

### Cash management

```text
Maintain $500 reserve
```

### Labor

```text
Hire one hand when >=15 active plants
```

### Land

```text
NE at $4k
SW at $8k
SE at $12k
```

### Animals

```text
Not implemented
```

### Fertilizer

```text
Not implemented
```

### Opponent modeling

```text
Not implemented
```

### Learning

```text
None
```

---

# 36. Conclusion

**Balanced Farmer v1** establishes the first controlled strategy for Kaggriculture.

Its central principle is:

> **Use wheat to maintain liquidity, melon to generate high-value revenue, protect existing crops, sell intelligently, and reinvest profits into productive capacity.**

The model intentionally avoids premature complexity.

That is important because Kaggriculture is not merely a farming problem. It is a combination of:

```text
                 KAGGRICULTURE
                       │
       ┌───────────────┼────────────────┐
       │               │                │
  Operations       Economics         Strategy
       │               │                │
  Scheduling       Pricing          Competition
  Labor            Capital          Opponent
  Production       Inventory        Adaptation
       │               │                │
       └───────────────┼────────────────┘
                       │
                    PROFIT
                       │
                     WIN
```

Therefore, **Balanced Farmer v1 should be treated as our control model**.

The next objective is not simply to make the code larger. It is to identify **which economic decisions actually matter**, test them independently, and progressively replace the baseline heuristics with strategies supported by simulation evidence.


In [ ]:
%%writefile submission.py

from kaggle_environments.envs.kaggriculture.kaggriculture import CROPS


# ============================================================
# BALANCED FARMER v1
# ============================================================
#
# Strategy:
#   - Wheat = stable cash flow + future animal feed
#   - Melon = high-value cash crop
#   - Water all crops
#   - Harvest mature crops
#   - Sell wheat readily
#   - Sell melons only when price is attractive
#   - Buy seeds when needed
#   - Hire one farm hand once the farm becomes sufficiently busy
#   - Expand land when cash reserves are sufficiently large
#   - No animals or fertilizer in v1
#
# This is intended to be a simple BASELINE, not an optimized agent.
# ============================================================


# -----------------------------
# Crop constants
# -----------------------------

WHEAT_SEED_COST = CROPS["WHEAT"]["seed"]
MELON_SEED_COST = CROPS["MELON"]["seed"]

WHEAT_MAX_YIELD_DAY = CROPS["WHEAT"]["max_yield_day"]
MELON_MAX_YIELD_DAY = CROPS["MELON"]["max_yield_day"]


# -----------------------------
# Simple economic parameters
# -----------------------------

# Melon is valuable, but its price can crash when we sell too much.
MELON_SELL_THRESHOLD = 200

# Sell melons in batches rather than dumping the whole inventory.
MELON_SELL_BATCH = 5

# Wheat is primarily a cash-flow crop in this version.
WHEAT_SELL_THRESHOLD = 25

# Keep some cash available rather than spending everything.
MIN_CASH_RESERVE = 500

# Start hiring once enough crops exist that one farmer becomes inefficient.
HIRE_PLANT_THRESHOLD = 15

# Simple land-expansion thresholds.
LAND_BUY_THRESHOLDS = {
    "NE": 4000,
    "SW": 8000,
    "SE": 12000,
}


# ============================================================
# Utility functions
# ============================================================

def _step_toward(fx, fy, tx, ty):
    """
    Return one movement action that moves the farmer toward
    the target tile using Manhattan distance.
    """

    if fx > tx:
        return "WEST"

    if fx < tx:
        return "EAST"

    if fy > ty:
        return "NORTH"

    if fy < ty:
        return "SOUTH"

    return None


def _manhattan_distance(x1, y1, x2, y2):
    return abs(x1 - x2) + abs(y1 - y2)


def _count_plants(farm):
    """
    Count all active plants on the farm.
    """

    count = 0
    board_size = len(farm["tiles"])

    for y in range(board_size):
        for x in range(board_size):

            tile = farm["tiles"][y][x]

            if isinstance(tile, dict) and tile.get("kind") == "PLANT":
                count += 1

    return count


def _count_empty_tiles(farm):
    """
    Count unlocked empty tiles.
    """

    count = 0
    board_size = len(farm["tiles"])

    for y in range(board_size):
        for x in range(board_size):

            tile = farm["tiles"][y][x]

            if tile is None:
                count += 1

    return count


def _find_nearest_target(farm, target_types, fx, fy):
    """
    Find the nearest tile whose (x, y, purpose) is one of
    the requested target types.

    target_types example:
        {"WATER", "HARVEST", "PLANT_WHEAT"}
    """

    board_size = len(farm["tiles"])
    candidates = []

    for y in range(board_size):
        for x in range(board_size):

            tile = farm["tiles"][y][x]

            # --------------------------------------------
            # Plants
            # --------------------------------------------

            if isinstance(tile, dict) and tile.get("kind") == "PLANT":

                crop = tile.get("crop")

                # Watering has high priority.
                if (
                    "WATER" in target_types
                    and not tile.get("watered_today", False)
                ):
                    candidates.append(
                        (
                            _manhattan_distance(fx, fy, x, y),
                            0,
                            x,
                            y,
                            "WATER",
                        )
                    )

                # Harvest wheat when mature.
                if (
                    "HARVEST_WHEAT" in target_types
                    and crop == "WHEAT"
                    and tile.get("yield_units", 0) > 0
                ):
                    candidates.append(
                        (
                            _manhattan_distance(fx, fy, x, y),
                            1,
                            x,
                            y,
                            "HARVEST",
                        )
                    )

                # Harvest melon when mature.
                if (
                    "HARVEST_MELON" in target_types
                    and crop == "MELON"
                    and tile.get("yield_units", 0) > 0
                ):
                    candidates.append(
                        (
                            _manhattan_distance(fx, fy, x, y),
                            1,
                            x,
                            y,
                            "HARVEST",
                        )
                    )

            # --------------------------------------------
            # Empty tile
            # --------------------------------------------

            elif tile is None:

                if "PLANT_WHEAT" in target_types:
                    candidates.append(
                        (
                            _manhattan_distance(fx, fy, x, y),
                            2,
                            x,
                            y,
                            "PLANT_WHEAT",
                        )
                    )

                if "PLANT_MELON" in target_types:
                    candidates.append(
                        (
                            _manhattan_distance(fx, fy, x, y),
                            2,
                            x,
                            y,
                            "PLANT_MELON",
                        )
                    )

    if not candidates:
        return None

    # First minimize distance, then action priority.
    candidates.sort(key=lambda z: (z[0], z[1]))

    return candidates[0]


def _find_best_farming_target(farm, seeds, fx, fy):
    """
    Decide what the farmer should work on next.

    Priority:
        1. Water plants
        2. Harvest mature crops
        3. Plant melon
        4. Plant wheat
    """

    # ----------------------------------------------------
    # 1. Water first.
    # ----------------------------------------------------

    target = _find_nearest_target(
        farm,
        {"WATER"},
        fx,
        fy,
    )

    if target:
        return target

    # ----------------------------------------------------
    # 2. Harvest mature wheat.
    # ----------------------------------------------------

    target = _find_nearest_target(
        farm,
        {"HARVEST_WHEAT"},
        fx,
        fy,
    )

    if target:
        return target

    # ----------------------------------------------------
    # 3. Harvest mature melon.
    # ----------------------------------------------------

    target = _find_nearest_target(
        farm,
        {"HARVEST_MELON"},
        fx,
        fy,
    )

    if target:
        return target

    # ----------------------------------------------------
    # 4. Plant melon.
    #
    # Melon gets preference because it is our high-value
    # cash crop.
    # ----------------------------------------------------

    if seeds.get("MELON", 0) > 0:

        target = _find_nearest_target(
            farm,
            {"PLANT_MELON"},
            fx,
            fy,
        )

        if target:
            return target

    # ----------------------------------------------------
    # 5. Plant wheat.
    # ----------------------------------------------------

    if seeds.get("WHEAT", 0) > 0:

        target = _find_nearest_target(
            farm,
            {"PLANT_WHEAT"},
            fx,
            fy,
        )

        if target:
            return target

    return None


# ============================================================
# Market management
# ============================================================

def _market_actions(obs, farm, private):
    """
    Decide what market orders to submit.

    Strategy:

    Wheat:
        Sell when the market is at or above its normal price.

    Melon:
        Sell only when price >= $200.
        Never dump the entire melon inventory at once.

    Seeds:
        Maintain a small supply of both wheat and melon.
    """

    market = []

    money = farm.get("money", 0)

    seeds = private.get("seeds", {})
    shed = private.get("shed", {})

    prices = (obs.get("market", {}) or {}).get("prices", {})

    wheat_price = prices.get("WHEAT", 0)
    melon_price = prices.get("MELON", 0)

    # --------------------------------------------------------
    # Sell wheat.
    #
    # Wheat is our stable cash-flow resource.
    # --------------------------------------------------------

    wheat_in_shed = shed.get("WHEAT", 0)

    if wheat_in_shed > 0 and wheat_price >= WHEAT_SELL_THRESHOLD:

        market.append(
            ["SELL", "WHEAT", wheat_in_shed]
        )

    # --------------------------------------------------------
    # Sell melon.
    #
    # Only sell a limited batch so that we don't intentionally
    # crash the market.
    # --------------------------------------------------------

    melon_in_shed = shed.get("MELON", 0)

    if melon_in_shed > 0 and melon_price >= MELON_SELL_THRESHOLD:

        sell_amount = min(
            melon_in_shed,
            MELON_SELL_BATCH,
        )

        market.append(
            ["SELL", "MELON", sell_amount]
        )

    # --------------------------------------------------------
    # Buy seeds.
    #
    # Maintain a small working supply rather than buying huge
    # quantities.
    # --------------------------------------------------------

    reserve_cash = MIN_CASH_RESERVE

    # Buy wheat seed when none are available.
    if (
        seeds.get("WHEAT", 0) == 0
        and money >= WHEAT_SEED_COST + reserve_cash
    ):
        market.append(
            ["BUY_SEED", "WHEAT", 1]
        )

    # Buy melon seed when none are available.
    if (
        seeds.get("MELON", 0) == 0
        and money >= MELON_SEED_COST + reserve_cash
    ):
        market.append(
            ["BUY_SEED", "MELON", 1]
        )

    return market


# ============================================================
# Land expansion
# ============================================================

def _land_actions(farm):
    """
    Buy land when the farm has accumulated enough cash.

    We only buy one quadrant at a time.
    """

    money = farm.get("money", 0)

    unlocked = set(
        farm.get("unlocked_quadrants", [])
    )

    orders = []

    # --------------------------------------------------------
    # NE
    # --------------------------------------------------------

    if (
        "NE" not in unlocked
        and money >= LAND_BUY_THRESHOLDS["NE"]
    ):
        orders.append(
            ["BUY_LAND", "NE"]
        )

    # --------------------------------------------------------
    # SW
    # --------------------------------------------------------

    elif (
        "SW" not in unlocked
        and money >= LAND_BUY_THRESHOLDS["SW"]
    ):
        orders.append(
            ["BUY_LAND", "SW"]
        )

    # --------------------------------------------------------
    # SE
    # --------------------------------------------------------

    elif (
        "SE" not in unlocked
        and money >= LAND_BUY_THRESHOLDS["SE"]
    ):
        orders.append(
            ["BUY_LAND", "SE"]
        )

    return orders


# ============================================================
# Farm-hand management
# ============================================================

def _should_hire(farm):
    """
    Hire one farm hand once the farm becomes sufficiently busy.

    We deliberately limit the baseline to ONE hand.
    """

    plants = _count_plants(farm)

    hires_today = farm.get("hires_today", 0)

    if hires_today >= 1:
        return False

    if plants >= HIRE_PLANT_THRESHOLD:
        return True

    return False


# ============================================================
# Farm-hand behavior
# ============================================================

def _hand_action(obs, farm, private, hand_index):
    """
    Simple farm-hand behavior.

    The hand primarily maintains plants.

    The baseline does not attempt sophisticated multi-agent
    task allocation.
    """

    hands = farm.get("hands", [])

    if hand_index >= len(hands):
        return ["PASS"]

    hx, hy = hands[hand_index]

    target = _find_nearest_target(
        farm,
        {"WATER", "HARVEST_WHEAT", "HARVEST_MELON"},
        hx,
        hy,
    )

    if not target:
        return ["PASS"]

    _, _, tx, ty, purpose = target

    # Already standing on target.
    if hx == tx and hy == ty:

        if purpose == "WATER":
            return ["WATER"]

        if purpose == "HARVEST":
            return ["HARVEST"]

    # Move toward target.
    step = _step_toward(
        hx,
        hy,
        tx,
        ty,
    )

    if step:
        return [step]

    return ["PASS"]


# ============================================================
# Main agent
# ============================================================

def balanced_farmer(obs):
    """
    Balanced Farmer v1.

    This is the baseline agent.
    """

    farms = obs.get("farms", [])

    player = obs.get("player", 0)

    private = obs.get("private", {}) or {}

    # --------------------------------------------------------
    # Safety check.
    # --------------------------------------------------------

    if not farms or player >= len(farms):

        return {
            "farmer": ["PASS"],
            "hands": [],
            "market": [],
        }

    farm = farms[player]

    board_size = len(
        farm.get("tiles", [])
    )

    if board_size == 0:

        return {
            "farmer": ["PASS"],
            "hands": [],
            "market": [],
        }

    # --------------------------------------------------------
    # Current farmer position.
    # --------------------------------------------------------

    fx, fy = farm.get(
        "farmer",
        [0, 0],
    )

    tile = farm["tiles"][fy][fx]

    seeds = private.get(
        "seeds",
        {},
    )

    day = obs.get(
        "day",
        0,
    )

    # --------------------------------------------------------
    # Market orders.
    # --------------------------------------------------------

    market = _market_actions(
        obs,
        farm,
        private,
    )

    # --------------------------------------------------------
    # Land expansion.
    # --------------------------------------------------------

    market.extend(
        _land_actions(farm)
    )

    # --------------------------------------------------------
    # Hire one farm hand when the farm is sufficiently busy.
    # --------------------------------------------------------

    if _should_hire(farm):

        market.append(
            ["HIRE"]
        )

    # --------------------------------------------------------
    # Farmer action.
    # --------------------------------------------------------

    farmer = ["PASS"]

    # ========================================================
    # If standing on a plant
    # ========================================================

    if (
        isinstance(tile, dict)
        and tile.get("kind") == "PLANT"
    ):

        crop = tile.get("crop")

        planted_day = tile.get(
            "planted_day",
            day,
        )

        age = day - planted_day

        # ----------------------------------------------------
        # Water immediately if needed.
        # ----------------------------------------------------

        if not tile.get(
            "watered_today",
            False,
        ):

            farmer = ["WATER"]

        # ----------------------------------------------------
        # Harvest wheat when mature.
        # ----------------------------------------------------

        elif (
            crop == "WHEAT"
            and age >= WHEAT_MAX_YIELD_DAY
            and tile.get("yield_units", 0) > 0
        ):

            farmer = ["HARVEST"]

        # ----------------------------------------------------
        # Harvest melon when mature.
        # ----------------------------------------------------

        elif (
            crop == "MELON"
            and age >= MELON_MAX_YIELD_DAY
            and tile.get("yield_units", 0) > 0
        ):

            farmer = ["HARVEST"]

        # ----------------------------------------------------
        # Otherwise move toward another useful tile.
        # ----------------------------------------------------

        else:

            target = _find_best_farming_target(
                farm,
                seeds,
                fx,
                fy,
            )

            if target:

                _, _, tx, ty, purpose = target

                step = _step_toward(
                    fx,
                    fy,
                    tx,
                    ty,
                )

                if step:
                    farmer = [step]

    # ========================================================
    # If standing on an empty tile
    # ========================================================

    elif tile is None:

        # Prefer melon when a melon seed is available.
        if seeds.get("MELON", 0) > 0:

            farmer = [
                "PLANT",
                "MELON",
            ]

        elif seeds.get("WHEAT", 0) > 0:

            farmer = [
                "PLANT",
                "WHEAT",
            ]

        else:

            target = _find_best_farming_target(
                farm,
                seeds,
                fx,
                fy,
            )

            if target:

                _, _, tx, ty, purpose = target

                step = _step_toward(
                    fx,
                    fy,
                    tx,
                    ty,
                )

                if step:
                    farmer = [step]

    # ========================================================
    # If standing on a weed or structure
    # ========================================================

    else:

        target = _find_best_farming_target(
            farm,
            seeds,
            fx,
            fy,
        )

        if target:

            _, _, tx, ty, purpose = target

            step = _step_toward(
                fx,
                fy,
                tx,
                ty,
            )

            if step:
                farmer = [step]

    # --------------------------------------------------------
    # Farm hands.
    # --------------------------------------------------------

    hands = []

    active_hands = farm.get(
        "hands",
        [],
    )

    for i in range(len(active_hands)):

        action = _hand_action(
            obs,
            farm,
            private,
            i,
        )

        hands.append(action)

    # --------------------------------------------------------
    # Return complete action.
    # --------------------------------------------------------

    return {
        "farmer": farmer,
        "hands": hands,
        "market": market,
    }


# ============================================================
# Kaggle entry point
# ============================================================

def agent(obs):
    return balanced_farmer(obs)